# Behavior Modeling API: BITS Implementation

## Introduction

Tactics2D provides unified reimplementations of a collection of representative traffic participant behavior models to support the development, validation, and testing of Autonomous Driving Systems (ADS) with realistic and scalable traffic interactions. BITS is one of the default behavior models integrated into Tactics2D.

Original paper: [BITS: Bi-level Imitation for Traffic Simulation](https://arxiv.org/abs/2208.12403)

Original code: [NVlabs/traffic-behavior-simulation](https://github.com/NVlabs/traffic-behavior-simulation)

BITS adopts a bi-level imitation learning architecture that decomposes vehicle behavior generation into two stages: a high-level spatial planner that samples goal candidates from a rasterized scene representation, and a low-level goal-conditional policy that generates trajectory candidates from the sampled goals. A multi-agent prediction head simultaneously forecasts neighboring agents' future states, enabling closed-loop collision filtering during candidate scoring. This hierarchical design combines the scalability of raster-based scene encoding with the precision of trajectory optimization.

The original paper validates BITS on the NuPlan dataset, demonstrating its capability for realistic traffic simulation across diverse urban driving scenarios. Rather than reproducing only these benchmark settings, Tactics2D reimplements BITS on top of its unified data representation, map abstraction, and scenario interface. Consequently, the behavior model can be executed on all datasets supported by Tactics2D, making BITS one of the framework's default foundational behavior models and providing a consistent behavior generation pipeline across heterogeneous traffic datasets.

In this documentation, we will demonstrate how to use the BITS behavior model in Tactics2D with both an in-memory synthetic scene and parsed dataset data.

## Environment Setup

Please install Tactics2D (`pip install 'tactics2[behavior]'`) or add the Tactics2D source directory to your `PYTHONPATH`. See the [Installation Guide](https://tactics2d.readthedocs.io/en/latest/installation/) for more details.

A pretrained BITS checkpoint is available at [](). Download the checkpoint and place it in a local directory. You will need to specify its path when initializing the model. The training scripts for BITS are provided in the repository [WoodOxen/tactics2d-behavior](https://github.com/WoodOxen/tactics2d-behavior).

## Dataset Preparation

Tactics2D does not require datasets to be stored in a fixed location. You can place a dataset in any directory and provide its path when loading it. Throughout this tutorial, **NuPlan** is used to demonstrate the standard workflow of BITS, while **InD, WOMD** serves as an additional example to show how BITS can be used with other datasets supported by Tactics2D.

## Use BITS for Behavior Generation

The pipeline below demonstrates a complete BITS prediction workflow. Every step that interacts with data (parsing, representing, predicting, and rendering) is handled by Tactics2D's public API. The notebook only provides glue code.

| Module | Key API |
|--------|---------|
| **Dataset parsers** | `parser.parse_trajectory(...)` → `(participants, time_range)` |
| **Map abstraction** | `map_.roadlines`, `map_.lanes`, `map_.areas` |
| **Participant model** | `participant.trajectory.get_state(frame)` |
| **Behavior model** | `model.predict(participants, map_, frame, agent_ids)` → `{agent_id: Trajectory}` |
| **BEVCamera** | `camera.update(frame, participants, ...)` → `geometry_data` |
| **MatplotlibRenderer** | `renderer.update(geometry_data)` |
| **Trajectory gradient** | `renderer.draw_gradient_trace(positions, colormap)` |
| **Color & style** | `participant.color = "purple"` overrides `COLOR_PALETTE` defaults |


In [1]:
from __future__ import annotations

import warnings

warnings.filterwarnings("ignore")

import logging

logging.basicConfig(level=logging.WARNING)

import matplotlib as mpl
import matplotlib.cm as cm
from matplotlib.animation import FuncAnimation
from shapely.geometry import Point
import numpy as np

from tactics2d.behavior.bits import BitsBehaviorModel
from tactics2d.dataset_parser import NuPlanParser, LevelXParser
from tactics2d.display.sensor import BEVCamera
from tactics2d.display.renderers import MatplotlibRenderer
from tactics2d.map.parser import OSMParser
from tactics2d.map.element import Map
from tactics2d.map.map_config import HIGHD_MAP_CONFIG, IND_MAP_CONFIG
from tactics2d.participant.element import Vehicle

pygame 2.6.1 (SDL 2.28.4, Python 3.9.25)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
mpl.rcParams.update(
    {
        "figure.dpi": 200,
        "font.family": "DejaVu Sans Mono",
        "font.size": 8,
        "animation.html": "html5",
        "animation.embed_limit": 100 * 1024 * 1024,
        "axes.edgecolor": "black",
        "axes.linewidth": 0.8,
        "axes.facecolor": "white",
        "figure.facecolor": "white",
    }
)

In [3]:
# ---- BITS model ----
CHECKPOINT_DIR = "../../checkpoints/bits"
DEVICE = "cuda"  # or "cpu"

model = BitsBehaviorModel.from_checkpoint(
    f"{CHECKPOINT_DIR}/bits_planner_epoch_0040.ckpt", device=DEVICE
)
config = model.config

print("history_steps:", config.history_steps)
print("future_steps:", config.future_steps)
print("dt:", config.dt, "s  (step_ms:", config.step_ms, "ms)")
print("raster_size:", config.raster_size, "pixels")
print("pixel_size:", config.pixel_size, "m/px")

# ---- Display constants ----
NUM_SECONDS = 10
PERCEPTION_RANGE = 200
VIEW_WIDTH = 120
VIEW_HEIGHT = 80

history_steps: 10
future_steps: 20
dt: 0.1 s  (step_ms: 100 ms)
raster_size: 224 pixels
pixel_size: 0.5 m/px


## Closed-Loop Usage

The following three steps demonstrate a reusable pipeline for visualizing BITS's predicted trajectories. Each step describes the relevant built-in modules provided by Tactics2D and how they are combined in the example code. For simplicity, this demo takes over only **one vehicle** at a time. To control multiple vehicles simultaneously, simply pass a list of agent IDs to the `model.predict()` function.

Unlike LimSim, BITS requires rasterized map context for inference (`include_raster=True` is enabled by default when loading via `load_bits_checkpoint`). BITS does not use lane-level route guidance, so no `route_map` argument is needed.

### Step 1: Select the Takeover Vehicle

`Vehicle` carries a `Trajectory` (a `dict[int, State]` mapping timestamps to positions) and a `color` attribute consumed by the renderer. `select_takeover_vehicle` picks the vehicle with the longest `history_states`; pass `ego_id` to `run_takeover_scenario` to override this. The color is resolved against `COLOR_PALETTE` — `"purple"` → `#8854d0`, the same mechanism that maps vehicle types to default colors.

In [4]:
def select_takeover_vehicle(participants):
    """Return the vehicle with the longest trajectory, or raise if none found."""
    vehicle_ids = [
        pid
        for pid, p in participants.items()
        if isinstance(p, Vehicle) and len(p.trajectory.history_states) > 0
    ]
    if not vehicle_ids:
        raise ValueError("No vehicle participant found.")
    return max(vehicle_ids, key=lambda pid: len(participants[pid].trajectory.history_states))

### Step 2: Predict Trajectories with BITS

`BitsBehaviorModel` wraps the bi-level architecture (spatial planner + trajectory module) behind a single call shared by all Tactics2D behavior models:

```python
predicted = model.predict(participants, map_, frame=frame, agent_ids=[ego_id])
```

- `participants` — the same dict from `parse_trajectory()`; each `Vehicle`'s history states up to `frame` are the model's input.
- `map_` — the `Map` from `parse_map()`, providing lane, roadline, and area geometry for rasterization.
- `agent_ids` — which participants to predict for; others are still observed as context.

Returns `{agent_id: Trajectory}` with predicted future states in world coordinates using the same `.get_state(frame)` API as history.

Internally, the model rasterizes the map around the ego agent, encodes the scene through a ResNet backbone, samples spatial goal candidates via a UNet decoder, and generates trajectory candidates conditioned on those goals. A closed-loop scorer filters candidates by likelihood, progress, lane adherence, and collision avoidance, returning the best plan.

### Step 3: Render with BEVCamera and MatplotlibRenderer

**BEVCamera** handles viewport culling, coordinate transforms, and geometry generation. `camera.update(...)` returns `(geometry_data, road_set, participant_set)` — the renderer consumes `geometry_data` directly.

**MatplotlibRenderer** draws the scene with automatic z-ordering (road areas < lines < buildings < vehicles < point clouds) and resolves colors from `COLOR_PALETTE` and `DEFAULT_COLOR` by element type.

**Trajectory gradient** (`renderer.enable_trajectory_gradient()` + `draw_gradient_trace()`) renders colormapped past/future traces at zorder 5 — near segments get the dark end, far segments the light end.

The `update()` closure below is called by `FuncAnimation` on every frame: clear old traces → collect active participants → get ego pose → `camera.update()` → `renderer.update()` → draw past trace (BuPu, reversed so closest point is first) → draw future prediction (GnBu).

In [5]:
def render_takeover_animation(
    participants, map_, playback_frames, ego_id, mpc_plans=None, resolution=(1200, 800)
):
    for roadline in map_.roadlines.values():
        if roadline.type_ is None:
            roadline.type_ = "roadline"
    camera = BEVCamera(id_=0, map_=map_, perception_range=PERCEPTION_RANGE)
    prev_road, prev_part = set(), set()
    renderer = MatplotlibRenderer(
        xlim=(-VIEW_WIDTH / 2, VIEW_WIDTH / 2),
        ylim=(-VIEW_HEIGHT / 2, VIEW_HEIGHT / 2),
        resolution=resolution,
        auto_scale=False,
    )
    renderer.enable_trajectory_gradient()
    ego_traj = participants[ego_id].trajectory
    plan_frames = sorted(mpc_plans.keys()) if mpc_plans else []

    def update(frame):
        nonlocal prev_road, prev_part
        renderer._remove_trajectory_lines()
        pids = [pid for pid, p in participants.items() if frame in p.trajectory.history_states]
        if ego_traj.has_state(frame):
            ego_pose = ego_traj.get_state(frame)
            cam_pos = Point(ego_pose.x, ego_pose.y)
            renderer.ax.set_xlim(ego_pose.x - VIEW_WIDTH / 2, ego_pose.x + VIEW_WIDTH / 2)
            renderer.ax.set_ylim(ego_pose.y - VIEW_HEIGHT / 2, ego_pose.y + VIEW_HEIGHT / 2)
        else:
            ego_pose = None
            cam_pos = Point(0.0, 0.0)
        gd, prev_road, prev_part = camera.update(
            frame, participants, pids, prev_road, prev_part, cam_pos
        )
        renderer.update(gd)
        if ego_pose is not None:
            past = [f for f in playback_frames if f <= frame and ego_traj.has_state(f)]
            if past:
                pts = [(ego_traj.get_state(f).x, ego_traj.get_state(f).y) for f in past]
                renderer.draw_gradient_trace(list(reversed(pts)), cm.BuPu)
            if plan_frames:
                cand = [f for f in plan_frames if f <= frame]
                if cand:
                    dxdy = mpc_plans[cand[-1]]
                    fpts = [(ego_pose.x, ego_pose.y)]
                    fpts.extend((ego_pose.x + dx, ego_pose.y + dy) for dx, dy in dxdy)
                    renderer.draw_gradient_trace(fpts, cm.GnBu)
        renderer.ax.set_title(
            f"BITS takeover: {ego_id}  |  frame {frame}  |  active: {len(pids)}", fontsize=7
        )

    return FuncAnimation(renderer.fig, update, frames=playback_frames, interval=100, repeat=True)

In [6]:
def run_takeover_scenario(
    parser,
    file_name=None,
    folder=None,
    map_file=None,
    map_folder=None,
    map_path=None,
    map_config=None,
    ego_id=None,
    ego_color="light-pink",
    fps=10,
    num_seconds=NUM_SECONDS,
    resolution=(1200, 800),
    replan_interval=20,
    **parse_kwargs,
):
    print("Parsing scenario ...")
    participants, time_range = parser.parse_trajectory(
        file=file_name, folder=folder, **parse_kwargs
    )

    # Map loading (same fallback chain as LimSim demo)
    map_ = None
    if hasattr(parser, "parse_map"):
        try:
            mf = map_file if map_file is not None else file_name
            mf_dir = map_folder if map_folder is not None else folder
            map_ = parser.parse_map(file=mf, folder=mf_dir, **parse_kwargs)
        except Exception:
            pass
    if map_ is None and map_path is not None:
        print(f"  loading map from {map_path}")
        map_ = OSMParser(lanelet2=True).parse(file_path=map_path, configs=map_config)
    if map_ is None:
        map_ = Map("empty_map", scenario_type="demo")

    print(f"  participants: {len(participants)},  frames: {time_range}")
    if ego_id is None:
        ego_id = select_takeover_vehicle(participants)
    participants[ego_id].color = ego_color
    print(f"  takeover target: {ego_id}  (color: {ego_color})")

    # ---- prepare takeover window ----
    ego = participants[ego_id]
    frames = sorted(ego.trajectory.history_states.keys())
    takeover_idx = min(int(fps * 1.0), len(frames) // 5)
    takeover = frames[takeover_idx]
    hist_n = takeover_idx

    orig_hist = dict(ego.trajectory.history_states)
    orig_frames = sorted(orig_hist.keys())

    def _truncate(at):
        ego.trajectory._history_states = {f: s for f, s in orig_hist.items() if f <= at}
        ego.trajectory._frames = sorted(ego.trajectory._history_states.keys())

    _truncate(takeover)

    # ---- MPC with stride: replan every replan_interval frames ----------
    total_future_frames = int(num_seconds * fps)
    mpc_xs, mpc_ys, mpc_ts, mpc_plans = [], [], [], {}
    cur = takeover
    steps_done = 0

    while steps_done < total_future_frames:
        try:
            pred = model.predict(participants, map_, cur, agent_ids=[ego_id])
        except Exception:
            break
        if ego_id not in pred:
            break

        pt = pred[ego_id]
        ff = sorted(f for f in pt.frames if f > cur)
        if not ff:
            break

        anchor = ego.trajectory.get_state(cur)
        mpc_plans[cur] = [(pt.get_state(f).x - anchor.x, pt.get_state(f).y - anchor.y) for f in ff]

        step_count = min(replan_interval, len(ff), total_future_frames - steps_done)
        for si in range(step_count):
            f = ff[si]
            s = pt.get_state(f)
            mpc_xs.append(s.x)
            mpc_ys.append(s.y)
            mpc_ts.append(f)
            if ego.trajectory.has_state(f):
                ego.trajectory._history_states[f] = s
            else:
                ego.trajectory.add_state(s)

        steps_done += step_count
        cur = ff[step_count - 1] if step_count < len(ff) else ff[-1]

    # ---- interpolate MPC to original frame numbers --------------------
    _truncate(takeover)
    target = [f for f in orig_frames if takeover < f][:total_future_frames]
    StateCls = type(ego.trajectory.get_state(takeover))

    if len(target) > 0 and len(mpc_ts) >= 1:
        if len(mpc_ts) >= 2:
            ix = np.interp(target, mpc_ts, np.array(mpc_xs))
            iy = np.interp(target, mpc_ts, np.array(mpc_ys))
        else:
            ix = np.full(len(target), mpc_xs[0])
            iy = np.full(len(target), mpc_ys[0])
        h = ego.trajectory.get_state(takeover).heading
        for i, (t, x, y) in enumerate(zip(target, ix, iy)):
            if i < len(target) - 1:
                dx, dy = ix[i + 1] - x, iy[i + 1] - y
                if abs(dx) > 1e-6 or abs(dy) > 1e-6:
                    h = np.arctan2(dy, dx)
            ego.trajectory.add_state(StateCls(frame=t, x=x, y=y, heading=float(h)))

    all_frames = frames[takeover_idx - hist_n : takeover_idx] + [
        f for f in target if f > frames[takeover_idx - 1]
    ]
    actual_seconds = len(target) / fps
    print(
        f"  playback: {len(all_frames)} frames  ({all_frames[0]}-{all_frames[-1]})  ~{actual_seconds:.1f}s  ({len(mpc_ts)} predicts)"
    )

    # ---- render ----
    ani = render_takeover_animation(
        participants, map_, all_frames, ego_id, mpc_plans=mpc_plans, resolution=resolution
    )
    return ani

### Example 1: NuPlan — Pittsburgh (Highway)

This example demonstrates BITS taking over a vehicle in a NuPlan training scenario from **Pittsburgh**. The trajectory is parsed from a `.db` file under `data/nuplan/data/cache/train_pittsburgh/`, while the map is loaded from a separate `.gpkg` file under `data/nuplan/maps/us-pa-pittsburgh-hazelwood/`. The ego vehicle (`id=43`) is traveling at **~15 m/s (54 km/h)** on a multilane road, providing a clear highway-following prediction task.

The ego vehicle is shown in pink, its ground-truth trajectory in purple, and the BITS prediction in blue.

Adjust the paths below to match your local data layout.

In [7]:
ani_nuplan = run_takeover_scenario(
    NuPlanParser(),
    file_name="2021.09.13.19.54.06_veh-45_00781_00843.db",
    folder="../../data/nuplan/data/cache/train_pittsburgh",
    map_file="map.gpkg",
    map_folder="../../data/nuplan/maps/us-pa-pittsburgh-hazelwood/9.17.1937",
    ego_id=43,
    fps=10,
)
ani_nuplan

Parsing scenario ...
  participants: 57,  frames: (22133298350, 22133360099)
  takeover target: 43  (color: light-pink)
  playback: 110 frames  (22133306300-22133311800)  ~10.0s  (100 predicts)


In [8]:
ani_vegas = run_takeover_scenario(
    NuPlanParser(),
    file_name="2021.05.18.21.31.22_veh-30_00062_00160.db",
    folder="../../data/nuplan/data/cache/train_vegas_1",
    map_file="map.gpkg",
    map_folder="../../data/nuplan/maps/us-nv-las-vegas-strip/9.15.1915",
    ego_id=181,
    fps=10,
)
ani_vegas

Parsing scenario ...
  participants: 285,  frames: (11943197049, 11943294999)
  takeover target: 181  (color: light-pink)
  playback: 110 frames  (11943240299-11943245799)  ~10.0s  (100 predicts)


In [9]:
ani_turn = run_takeover_scenario(
    NuPlanParser(),
    file_name="2021.08.26.18.24.36_veh-28_00578_00663.db",
    folder="../../data/nuplan/data/cache/train_boston",
    map_file="map.gpkg",
    map_folder="../../data/nuplan/maps/us-ma-boston/9.12.1817",
    ego_id=69,
    fps=10,
)
ani_turn

Parsing scenario ...
  participants: 343,  frames: (20572546799, 20572631750)
  takeover target: 69  (color: light-pink)
  playback: 110 frames  (20572599900-20572605400)  ~10.0s  (100 predicts)


### Example 2: HighD (LevelX) — Location 1, Recording 01

HighD is parsed using `LevelXParser("highD")` at 25 Hz based on German highway recordings. The corresponding Lanelet2 `.osm` map is parsed with `OSMParser` using `HIGHD_MAP_CONFIG`. Recording 11 corresponds to location 1 (`highD_1`). Due to the high-speed highway environment (approximately 30 m/s), the generated trajectories are considerably longer than those in urban datasets such as InD.

Adjust the paths below according to your local data organization.

This scenario represents an out-of-distribution case compared with the NuPlan training data. Since BITS learns behavioral patterns from the training distribution, it shows limited generalization to high-speed highway scenarios. As a result, the predicted trajectory deviates from the lane and fails to maintain proper lane-following behavior. The model is expected to perform better on scenarios that are closer to its training distribution.

In [10]:
ani_highd = run_takeover_scenario(
    LevelXParser("highD"),
    file_name=11,
    folder="../../data/highD/data",
    map_path="../../data/highD_map/highD_1.osm",
    map_config=HIGHD_MAP_CONFIG["highD_1"],
    ego_id=18,
    fps=25,
)
ani_highd

Parsing scenario ...


  loading map from ../../data/highD_map/highD_1.osm
  participants: 1776,  frames: (np.int64(40), np.int64(611080))
  takeover target: 18  (color: light-pink)
  playback: 240 frames  (40-9640)  ~8.6s  (250 predicts)


### Example 3: inD (LevelX) — Location 1, Recording 00

This example demonstrates BITS on the **inD** dataset using `LevelXParser("inD")`, which processes the data at **25 Hz**. The road network is loaded from a Lanelet2 `.osm` map using `OSMParser` together with `IND_MAP_CONFIG`. Recording **00** corresponds to **Location 1** (`inD_1`).

In this scenario, the ego vehicle navigates an intersection where surrounding vehicles create ambiguous interaction patterns. BITS's spatial planner samples multiple goal candidates, and the closed-loop scorer selects the safest option based on collision and lane-adherence costs.

Adjust the paths below to match your local data layout.

In [11]:
ani_ind = run_takeover_scenario(
    LevelXParser("inD"),
    file_name=7,
    folder="../../data/inD/data",
    map_path="../../data/inD_map/inD_1.osm",
    map_config=IND_MAP_CONFIG["inD_1"],
    ego_id=12,
    fps=25,
)
ani_ind

Parsing scenario ...
  loading map from ../../data/inD_map/inD_1.osm
  participants: 212,  frames: (np.int64(0), np.int64(1055240))
  takeover target: 12  (color: light-pink)


  playback: 161 frames  (10240-16680)  ~5.4s  (250 predicts)


## Parameter Impact Analysis

The following tables summarise how `BitsConfig` parameters affect runtime and behavior. Adjust these based on your accuracy-vs-speed trade-off.

### Runtime-dominant Parameters

| Parameter | Effect | Guidance |
|-----------|--------|----------|
| `history_steps` | Controls how many past frames the policy observes. More history = larger input tensors and slower rasterization. | 10 for quick previews; 20–30 for richer context. Must match the trained checkpoint. |
| `future_steps` | Number of predicted future states (planning horizon). Longer horizons increase the policy decoder rollout. | 20 (2 s) for quick previews; 50–80 (5–8 s) for closed-loop simulation. Must match the trained checkpoint. |
| `raster_size` | Raster image resolution in pixels. Larger rasters increase CNN inference cost quadratically. | 224 for standard use; 112 for speed, 448 for detail. Must match the trained checkpoint. |
| `pixel_size` | Physical size of one raster pixel in meters. Together with `raster_size`, determines the physical coverage of the raster image (`raster_size` × `pixel_size` = 112 m at defaults). | 0.5 m for standard use. Must match the trained checkpoint. |
| `max_agents` | Caps the number of neighboring agents in the rasterized scene. More agents = larger input tensors. | 20 for urban scenarios; reduce to 10 to speed up batching. |
| `max_agents_distance` | Maximum distance for neighbor consideration. Smaller values exclude distant agents. | 30 m for urban; reduce to 15–20 m to shrink agent context. |
| `include_non_vehicle_neighbors` | Whether to include pedestrians, cyclists, and other non-vehicle participants as observed neighbors. | `False` by default. Enable for mixed-traffic scenarios. |
| `default_vehicle_length` | Fallback vehicle length when the participant does not provide one. Affects rasterized footprint and collision detection. | 4.8 m (default). Adjust to match your scenario's typical vehicle size. |
| `default_vehicle_width` | Fallback vehicle width when the participant does not provide one. Affects rasterized footprint and collision detection. | 1.9 m (default). Adjust to match your scenario's typical vehicle size. |

### Behavior-dominant Parameters

| Parameter | Effect | Guidance |
|-----------|--------|----------|
| `collision_weight` | Penalty weight for predicted collisions in candidate scoring. Higher values produce more conservative behavior. | 100–500. Increase if agents drive too aggressively. |
| `lane_weight` | Penalty for drivable area violations. Higher values keep agents more strictly on the road. | 50–200. Increase for narrow roads or strict lane-keeping. |
| `progress_weight` | Reward for forward progress along the route. Higher values encourage faster, more goal-directed driving. | 1.0–5.0. Increase for highway, decrease for dense intersections. |
| `likelihood_weight` | Weight for the learned policy's own score. Balances learned behavior against rule-based costs. | 1.0 (default). Decrease to rely more on hand-crafted costs. |
| `drivable_distance_clip` | Maximum distance penalty (in meters) for a trajectory point falling outside the drivable area. Caps the per-point lane violation cost. | 10.0 m (default). Increase to penalize off-road deviations more severely. |
| `dynamics_speed_max` | Maximum allowed speed in m/s. Clips trajectory speeds during scoring. | 20–30 m/s for highway; 10–15 m/s for urban. |
| `dynamics_speed_min` | Minimum allowed speed in m/s. Negative values permit reverse driving. | -10.0 m/s (default). Set to 0.0 to forbid reversing. |
| `dynamics_acceleration_min` | Maximum deceleration in m/s². Controls braking aggressiveness. | -6.0 to -4.0. More negative = harder braking allowed. |
| `dynamics_acceleration_max` | Maximum acceleration in m/s². Controls throttle aggressiveness. | 3.0–5.0. Higher = more aggressive acceleration. |
| `dynamics_max_steer` | Maximum steering angle in radians. Limits how sharply the vehicle can turn. | 0.5 rad (≈28.6°) default. Reduce for smoother highway driving; increase for tight urban turns. |
| `dynamics_max_yawvel` | Maximum yaw velocity in rad/s. Limits how quickly the vehicle can change heading. | 8.0 rad/s (default). Reduce for smoother, more stable trajectories. |

### Quick Configurations

| Use Case | Key Settings |
|----------|-------------|
| **Fast preview** | `history_steps=10`, `future_steps=20`, `raster_size=112`, `max_agents=10` |
| **Balanced demo** (current) | `history_steps=10`, `future_steps=20`, `raster_size=224`, `max_agents=20` |
| **High-fidelity eval** | `history_steps=30`, `future_steps=50`, `raster_size=224`, `max_agents=32` |


!!! warning

    The following parameters **must** match the checkpoint used for training:

    - `history_steps`
    - `future_steps`
    - `raster_size`
    - `pixel_size`

    Changing any of them without retraining will cause shape mismatches.

    The **Fast preview** and **High-fidelity eval** presets therefore require separately trained checkpoints.

    The remaining parameters (`*_weight`, `dynamics_*`, `max_agents`, etc.) can be modified freely during inference.